# P5 — Intelligent Fine-Grained Vehicle Recognition & Analysis System
**Stanford Cars (196 classes) · Classical CV + YOLOv8 detection + ConvNeXt fine-grained classification + hybrid fusion**

This notebook trains and evaluates everything the problem statement asks for, then exports a ready-to-use `artifacts/` folder (models + `inference.py`) for your frontend.

| Section | What it covers (PS requirement) |
|---|---|
| 0–2 | Setup, dataset download/parsing (make / model / year split), EDA |
| 3 | **Classical CV**: preprocessing & normalisation, illumination correction, edges, contours, ROI, colour/texture/HOG/geometric features (§3A) |
| 4 | **Classical-feature classifiers** (LogReg, RBF-SVM) — baseline for comparison (§4.4) |
| 5 | **Detection**: classical contour ROI vs pretrained YOLOv8 vs fine-tuned YOLOv8 → IoU, mAP@0.5, mAP@0.5:0.95 (§3B) |
| 6 | **Fine-grained classifier**: ConvNeXt-Tiny transfer learning on 196 classes with illumination / viewpoint / occlusion / resolution augmentation (§3B) |
| 7–8 | Unseen-test evaluation (Top-1/Top-5, make / make+model accuracy), **hybrid integration + ablation** (§3C, §4.4, §4.5) |
| 9 | **Robustness**: illumination, occlusion, resolution, blur, noise, rotation, viewpoint slices (§4.2, §4.3) |
| 10 | **Confidence**: calibration, threshold analysis, low-confidence handling (§3C, §5) |
| 11 | **Optimisation**: FP16, ONNX, INT8 quantisation, pruning → size & latency (§4.6) |
| 12 | **Failure cases** with automatic cause tagging + Grad-CAM (§4.8) |
| 13–14 | End-to-end `VehicleRecognizer`, upload demo, artifact export for frontend |

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better). Outputs and checkpoints go to Google Drive (`MyDrive/P5_vehicle_recognition`), so a disconnect doesn't lose training — just re-run all cells; finished stages are skipped and the classifier resumes from its last epoch.

Approximate time on a T4: data + classical ≈ 20 min · YOLO fine-tune ≈ 30 min · classifier ≈ 50–60 min · evaluation/robustness/optimisation ≈ 30 min. Set `QUICK = True` in the config cell for a ~20 min smoke test first.

In [ ]:
# @title 0. Runtime check + installs
!nvidia-smi -L || echo "NO GPU — switch runtime to T4 GPU"
!pip -q install ultralytics timm onnx onnxruntime onnxscript scikit-image

In [ ]:
# @title 1. Imports, config, Drive
import os, sys, json, time, math, random, glob, shutil, copy, warnings, subprocess
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from scipy.special import log_softmax, softmax
cv2.setNumThreads(0)

QUICK = False   # True = fast smoke test (few epochs, test subset)

CFG = dict(
    SEED=42, VAL_FRAC=0.10,
    CROP_PAD=0.08,            # context padding around the vehicle box
    CACHE_MAX_SIDE=448,       # cached crop size (speeds up training)
    CLS_MODEL='convnext_tiny.fb_in22k_ft_in1k',
    IMG_SIZE=288, BATCH=32, EPOCHS=20, LR=2e-4, HEAD_LR_MULT=5.0,
    WEIGHT_DECAY=0.05, LABEL_SMOOTH=0.1, WARMUP_EPOCHS=1, NUM_WORKERS=2,
    YOLO_BASE='yolov8s.pt', YOLO_EPOCHS=15, YOLO_IMGSZ=640, YOLO_BATCH=16,
    DET_CONF=0.25,
    N_ROBUST=1000, N_OPT_EVAL=1000,
    TARGET_PRECISION=0.95,    # accuracy required on "confident" predictions
    DARK_THR=80, BRIGHT_THR=200,
)
if QUICK:
    CFG.update(EPOCHS=2, YOLO_EPOCHS=1, N_ROBUST=200, N_OPT_EVAL=200)

SEED = CFG['SEED']
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NW = CFG['NUM_WORKERS']
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/P5_vehicle_recognition'
except Exception:
    OUT = '/content/P5_vehicle_recognition'
WORK = '/content/work'
ART, PLOTS, CKPT = f'{OUT}/artifacts', f'{OUT}/plots', f'{OUT}/checkpoints'
for d in [WORK, ART, PLOTS, CKPT]:
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, WORK)

RESULTS_PATH = f'{ART}/metrics.json'
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
def _js(o):
    if hasattr(o, 'item'): return o.item()
    if hasattr(o, 'tolist'): return o.tolist()
    return str(o)
def save_results():
    json.dump(RESULTS, open(RESULTS_PATH, 'w'), indent=2, default=_js)
def savefig(name):
    plt.savefig(f'{PLOTS}/{name}.png', dpi=130, bbox_inches='tight')
print('Outputs ->', OUT)

## 2. Dataset — Stanford Cars (GitHub mirror)
Images + `.mat` annotations (bbox `x1,y1,x2,y2` and class 1–196). Class names are split into **Make / Model / Year** (e.g. `Aston Martin | V8 Vantage Convertible | 2012`).

In [ ]:
# @title 2.1 Download + locate files
DATA_REPO = '/content/StanfordCars-Dataset'
if not os.path.exists(DATA_REPO):
    !git clone --depth 1 https://github.com/jhpohovey/StanfordCars-Dataset.git {DATA_REPO}

def find_file(name):
    hits = glob.glob(f'{DATA_REPO}/**/{name}', recursive=True)
    assert hits, f'{name} not found'
    return hits[0]
def find_img_dir(name):
    for d in glob.glob(f'{DATA_REPO}/**/{name}', recursive=True):
        if os.path.isdir(d) and glob.glob(d + '/*.jpg'):
            return d
    raise FileNotFoundError(name)

META_MAT = find_file('cars_meta.mat')
TRAIN_MAT = find_file('cars_train_annos.mat')
TEST_MAT = find_file('cars_test_annos_withlabels.mat')
TRAIN_IMG_DIR, TEST_IMG_DIR = find_img_dir('cars_train'), find_img_dir('cars_test')

# Git-LFS guard: if images are LFS pointers, pull the real files
_probe = sorted(glob.glob(TRAIN_IMG_DIR + '/*.jpg'))[0]
if cv2.imread(_probe) is None:
    subprocess.run(f'cd {DATA_REPO} && git lfs install && git lfs pull', shell=True)
assert cv2.imread(_probe) is not None, 'Images unreadable'
print(TRAIN_IMG_DIR, len(os.listdir(TRAIN_IMG_DIR)), '|', TEST_IMG_DIR, len(os.listdir(TEST_IMG_DIR)))

In [ ]:
# @title 2.2 Parse annotations, make/model/year, train/val/test split
import scipy.io as sio
from sklearn.model_selection import train_test_split

MULTI_WORD_MAKES = ['AM General', 'Aston Martin', 'Land Rover']
def parse_name(n):
    parts = n.split()
    year = parts[-1] if parts[-1].isdigit() else ''
    body = ' '.join(parts[:-1] if year else parts)
    for mk in MULTI_WORD_MAKES:
        if body.startswith(mk + ' '):
            return mk, body[len(mk) + 1:], year
    mk, _, rest = body.partition(' ')
    return mk, rest, year

CLASS_NAMES = [str(x) for x in sio.loadmat(META_MAT, squeeze_me=True)['class_names']]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_META = []
for i, n in enumerate(CLASS_NAMES):
    mk, md, yr = parse_name(n)
    CLASS_META.append(dict(id=i, name=n, make=mk, model=md, year=yr))
MAKES = sorted({c['make'] for c in CLASS_META})
MAKE_ID = np.array([MAKES.index(c['make']) for c in CLASS_META])
MM_NAMES = sorted({c['make'] + ' ' + c['model'] for c in CLASS_META})
MM_ID = np.array([MM_NAMES.index(c['make'] + ' ' + c['model']) for c in CLASS_META])
print(f'{NUM_CLASSES} classes | {len(MAKES)} makes | {len(MM_NAMES)} make+model groups')

def load_annos(path, img_dir):
    a = sio.loadmat(path, squeeze_me=True)['annotations']
    df = pd.DataFrame({
        'fname': [os.path.basename(str(f)) for f in a['fname']],
        'x1': a['bbox_x1'].astype(float), 'y1': a['bbox_y1'].astype(float),
        'x2': a['bbox_x2'].astype(float), 'y2': a['bbox_y2'].astype(float),
        'label': a['class'].astype(int) - 1})
    df['path'] = img_dir + '/' + df.fname
    sizes = [Image.open(p).size for p in tqdm(df.path, desc='sizes', leave=False)]
    df['W'] = [s[0] for s in sizes]; df['H'] = [s[1] for s in sizes]
    return df

full_train_df = load_annos(TRAIN_MAT, TRAIN_IMG_DIR)
test_df = load_annos(TEST_MAT, TEST_IMG_DIR)
train_df, val_df = train_test_split(full_train_df, test_size=CFG['VAL_FRAC'],
                                    stratify=full_train_df.label, random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
if QUICK:
    test_df = test_df.sample(1000, random_state=SEED)
test_df = test_df.reset_index(drop=True)
print('train', len(train_df), '| val', len(val_df), '| test (unseen)', len(test_df))
GT_TEST = test_df[['x1', 'y1', 'x2', 'y2']].values

def cls_metrics(scores, y):
    """Top-1/Top-5 plus hierarchical accuracies (make, make+model ignoring year)."""
    scores, y = np.asarray(scores), np.asarray(y)
    pred = scores.argmax(1); top5 = np.argsort(-scores, 1)[:, :5]
    return dict(top1=float((pred == y).mean()), top5=float((top5 == y[:, None]).any(1).mean()),
                make_acc=float((MAKE_ID[pred] == MAKE_ID[y]).mean()),
                make_model_acc=float((MM_ID[pred] == MM_ID[y]).mean()))
pd.DataFrame(CLASS_META).head(8)

In [ ]:
# @title 2.3 EDA
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
full_train_df.label.value_counts().sort_index().plot(ax=ax[0]); ax[0].set_title('Train images per class')
ax[1].scatter(full_train_df.W, full_train_df.H, s=2, alpha=.3); ax[1].set_title('Image resolution (W vs H)')
ar = (full_train_df.x2 - full_train_df.x1) / (full_train_df.y2 - full_train_df.y1)
ax[2].hist(ar, 60); ax[2].set_title('BBox aspect ratio (viewpoint proxy)')
savefig('eda_stats'); plt.show()
yr = pd.Series([CLASS_META[l]['year'] for l in full_train_df.label]).value_counts().sort_index()
print('Images per manufacturing year:\n', yr.to_dict())

fig, axs = plt.subplots(2, 4, figsize=(18, 7))
for ax_, (_, r) in zip(axs.ravel(), full_train_df.sample(8, random_state=1).iterrows()):
    im = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    cv2.rectangle(im, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), (0, 255, 0), 3)
    ax_.imshow(im); ax_.set_title(CLASS_NAMES[r.label], fontsize=9); ax_.axis('off')
savefig('eda_samples'); plt.show()

## 3. Classical Computer Vision module
Written to `classical_cv.py` (re-used by training, evaluation, and the exported `inference.py`):
- **Preprocessing & normalisation**: aspect-preserving resize, per-channel z-score, adaptive illumination correction (gamma + CLAHE on LAB-L for dark / washed-out images)
- **Edge & contour analysis**: Gaussian blur → auto-threshold Canny → morphological closing → external contours
- **ROI extraction**: classical vehicle localiser (largest central, car-shaped contour) + box cropping with context padding
- **Colour features**: HSV histograms + LAB colour moments · **Texture**: multi-scale LBP + GLCM statistics · **Shape**: HOG
- **Geometric features**: aspect ratio, edge density, contour area ratio / extent / solidity / circularity, Hu moments, edge-orientation histogram, left–right symmetry

In [ ]:
%%writefile /content/work/classical_cv.py
"""Classical computer-vision module for P5 fine-grained vehicle recognition."""
import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

HOG_SIZE = (160, 96)  # (W, H) — cars are wider than tall


# ---------------- preprocessing / normalisation ----------------
def to_bgr(img):
    """PIL image, RGB array (from PIL) or BGR array (from cv2) / file path -> BGR uint8."""
    if isinstance(img, str):
        im = cv2.imread(img)
        if im is None:
            raise ValueError(f'Cannot read {img}')
        return im
    if hasattr(img, 'mode'):
        img = np.array(img.convert('RGB'))[:, :, ::-1]
    img = np.ascontiguousarray(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img


def resize_max(bgr, max_side):
    h, w = bgr.shape[:2]
    s = max_side / max(h, w)
    if s < 1:
        bgr = cv2.resize(bgr, (max(1, int(round(w * s))), max(1, int(round(h * s)))), interpolation=cv2.INTER_AREA)
    return bgr


def apply_clahe(bgr, clip=2.0, grid=8):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)


def brightness(bgr):
    return float(cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[:, :, 2].mean())


def adaptive_enhance(bgr, dark_thr=80, bright_thr=200):
    """Illumination normalisation: gamma-lift + CLAHE for dark images, CLAHE for washed-out ones."""
    v = brightness(bgr)
    if v < dark_thr:
        gamma = max(0.4, v / dark_thr)
        lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
        return apply_clahe(cv2.LUT(bgr, lut), clip=2.5), 'dark: gamma+CLAHE'
    if v > bright_thr:
        return apply_clahe(bgr, clip=1.5), 'bright: CLAHE'
    return bgr, 'none'


def normalize_image(bgr):
    """Per-channel z-score normalisation; returns float image and a displayable uint8 version."""
    f = bgr.astype(np.float32)
    mean, std = f.reshape(-1, 3).mean(0), f.reshape(-1, 3).std(0) + 1e-6
    z = (f - mean) / std
    disp = cv2.normalize(z, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return z, disp


# ---------------- edges / contours / ROI ----------------
def auto_canny(gray, sigma=0.33):
    v = float(np.median(gray))
    return cv2.Canny(gray, int(max(0, (1 - sigma) * v)), int(min(255, (1 + sigma) * v)))


def edge_contour_analysis(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = auto_canny(blur)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, k, iterations=2)
    closed = cv2.dilate(closed, k, iterations=1)
    cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return dict(gray=gray, blur=blur, edges=edges, closed=closed, contours=cnts)


def classical_roi(img, max_side=512):
    """Classical vehicle localisation: CLAHE -> Canny -> closing -> best (large, central, car-shaped) contour.
    Returns (box_xyxy in original coords or None, score in [0,1], debug dict)."""
    bgr = to_bgr(img)
    H, W = bgr.shape[:2]
    small = apply_clahe(resize_max(bgr, max_side))
    s = small.shape[1] / W
    ea = edge_contour_analysis(small)
    h, w = small.shape[:2]
    best, best_score = None, 0.0
    for c in ea['contours']:
        x, y, cw, ch = cv2.boundingRect(c)
        area = cw * ch / (w * h)
        if area < 0.05:
            continue
        cx, cy = (x + cw / 2) / w, (y + ch / 2) / h
        centrality = 1 - min(1.0, np.hypot(cx - 0.5, cy - 0.5) / 0.71)
        ar = cw / max(ch, 1)
        ar_score = 1.0 if 0.8 <= ar <= 3.5 else 0.5
        score = area * (0.5 + 0.5 * centrality) * ar_score
        if score > best_score:
            best_score, best = score, (x, y, x + cw, y + ch)
    if best is None:
        return None, 0.0, ea
    return tuple(v / s for v in best), float(min(1.0, best_score)), ea


def crop_box(bgr, box, pad=0.0):
    if box is None:
        return bgr
    H, W = bgr.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in box]
    bw, bh = x2 - x1, y2 - y1
    x1, y1 = int(max(0, x1 - pad * bw)), int(max(0, y1 - pad * bh))
    x2, y2 = int(min(W, x2 + pad * bw)), int(min(H, y2 + pad * bh))
    if x2 - x1 < 8 or y2 - y1 < 8:
        return bgr
    return bgr[y1:y2, x1:x2]


# ---------------- feature extraction ----------------
def color_features(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    feats = []
    for ch, bins, rng in [(0, 18, 180), (1, 8, 256), (2, 8, 256)]:
        h = cv2.calcHist([hsv], [ch], None, [bins], [0, rng]).ravel()
        feats.append(h / (h.sum() + 1e-6))
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    feats.append(np.concatenate([lab.mean(0), lab.std(0)]) / 255.0)
    return np.concatenate(feats)


def texture_features(gray):
    g = cv2.resize(gray, (128, 128))
    feats = []
    for P, R in [(8, 1), (16, 2)]:
        lbp = local_binary_pattern(g, P, R, method='uniform')
        h, _ = np.histogram(lbp, bins=P + 2, range=(0, P + 2))
        feats.append(h / (h.sum() + 1e-6))
    q = (g // 4).astype(np.uint8)
    glcm = graycomatrix(q, distances=[1, 3], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                        levels=64, symmetric=True, normed=True)
    for p in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        feats.append(graycoprops(glcm, p).ravel())
    return np.concatenate(feats)


def hog_features(gray, visualize=False):
    g = cv2.resize(gray, HOG_SIZE)
    return hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2),
               block_norm='L2-Hys', visualize=visualize, feature_vector=True)


def geometric_features(bgr):
    h, w = bgr.shape[:2]
    ea = edge_contour_analysis(resize_max(bgr, 256))
    edges, cnts = ea['edges'], ea['contours']
    hh, ww = edges.shape
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        area = cv2.contourArea(c)
        x, y, cw, ch = cv2.boundingRect(c)
        hull_a = cv2.contourArea(cv2.convexHull(c)) + 1e-6
        per = cv2.arcLength(c, True) + 1e-6
        f = [area / (hh * ww), area / (cw * ch + 1e-6), area / hull_a, 4 * np.pi * area / per ** 2, cw / max(ch, 1)]
        hu = cv2.HuMoments(cv2.moments(c)).ravel()
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    else:
        f, hu = [0.0] * 5, np.zeros(7)
    gx = cv2.Sobel(ea['blur'], cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(ea['blur'], cv2.CV_32F, 0, 1)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    oh, _ = np.histogram(ang % 180, bins=8, range=(0, 180), weights=mag)
    oh = oh / (oh.sum() + 1e-6)
    col = edges.mean(0)
    sym = float(np.clip(1 - np.abs(col - col[::-1]).mean() / (2 * col.mean() + 1e-6), 0, 1))
    # layout: [aspect, edge_density, n_contours/50, area_ratio, extent, solidity, circularity, contour_ar, hu*7, orient*8, symmetry]
    return np.concatenate([[w / max(h, 1), edges.mean() / 255.0, len(cnts) / 50.0], f, hu, oh, [sym]])


def extract_classical_features(img):
    """Full hand-crafted descriptor (~1.75k dims): HOG + colour + texture + geometry."""
    bgr = resize_max(to_bgr(img), 320)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    f = np.concatenate([hog_features(gray), color_features(bgr), texture_features(gray), geometric_features(bgr)])
    return np.nan_to_num(f.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)


# multiprocessing helpers (top-level so they pickle)
def features_from_path(args):
    path, box, pad = args
    bgr = cv2.imread(path)
    return extract_classical_features(crop_box(bgr, box, pad))


def roi_from_path(path):
    box, score, _ = classical_roi(cv2.imread(path))
    return box, score


# ---------------- visualisation of intermediate stages ----------------
def _rgb(im):
    return cv2.cvtColor(im, cv2.COLOR_GRAY2RGB) if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)


def _to_u8(a):
    a = a.astype(np.float32)
    a = (a - a.min()) / (a.max() - a.min() + 1e-6)
    return (a * 255).astype(np.uint8)


def processing_stages(img, box=None, max_side=640):
    """Returns ({stage_name: RGB uint8 image}, {feature_name: value}) for the app's 'intermediate results' panel.
    `box` (optional) is the detector box in ORIGINAL image coordinates."""
    src = to_bgr(img)
    W0 = src.shape[1]
    bgr = resize_max(src, max_side)
    s = bgr.shape[1] / W0
    st = {'Original': bgr}
    enh, mode = adaptive_enhance(bgr)
    st[f'Illumination normalisation ({mode})'] = enh
    _, nd = normalize_image(enh)
    st['Z-score normalised'] = nd
    ea = edge_contour_analysis(enh)
    st['Grayscale + Gaussian blur'] = ea['blur']
    st['Canny edges'] = ea['edges']
    st['Morphological closing'] = ea['closed']
    cimg = bgr.copy()
    cv2.drawContours(cimg, ea['contours'], -1, (0, 255, 0), 1)
    roi, rscore, _ = classical_roi(bgr)
    if roi is not None:
        cv2.rectangle(cimg, (int(roi[0]), int(roi[1])), (int(roi[2]), int(roi[3])), (255, 0, 0), 2)
    dbox = [v * s for v in box] if box is not None else None
    if dbox is not None:
        cv2.rectangle(cimg, (int(dbox[0]), int(dbox[1])), (int(dbox[2]), int(dbox[3])), (0, 0, 255), 2)
    st['Contours + ROI (blue=classical, red=detector)'] = cimg
    use = dbox if dbox is not None else roi
    crop = crop_box(bgr, use, 0.05) if use is not None else bgr
    st['Region of interest'] = crop
    g = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), (320, 192))
    _, hv = hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True)
    st['HOG (shape / gradients)'] = _to_u8(hv)
    st['LBP texture'] = _to_u8(local_binary_pattern(g, 8, 1, 'uniform'))
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hue = np.dstack([hsv[:, :, 0], np.full_like(hsv[:, :, 0], 255), np.full_like(hsv[:, :, 0], 255)])
    st['Hue map (colour)'] = cv2.cvtColor(hue, cv2.COLOR_HSV2BGR)
    gf = geometric_features(crop)
    c = crop[crop.shape[0] // 4: 3 * crop.shape[0] // 4, crop.shape[1] // 4: 3 * crop.shape[1] // 4]
    info = dict(brightness=round(brightness(bgr), 1), illumination_fix=mode,
                roi_aspect_ratio=round(float(gf[0]), 3), edge_density=round(float(gf[1]), 4),
                num_contours=len(ea['contours']), contour_solidity=round(float(gf[5]), 3),
                lr_symmetry=round(float(gf[-1]), 3),
                mean_rgb=[int(v) for v in c.reshape(-1, 3).mean(0)[::-1]],
                classical_roi_score=round(rscore, 3))
    return {k: _rgb(v) for k, v in st.items()}, info

In [ ]:
# @title 3.1 Import module + visualise every classical stage
import importlib, classical_cv
importlib.reload(classical_cv)
from classical_cv import *
shutil.copy(f'{WORK}/classical_cv.py', ART)

def show_stages(img, box=None, title=''):
    st, info = processing_stages(img, box)
    n = len(st); cols = 4; rows = math.ceil(n / cols)
    fig, axs = plt.subplots(rows, cols, figsize=(18, 3.6 * rows))
    for ax_, (k, v) in zip(axs.ravel(), st.items()):
        ax_.imshow(v); ax_.set_title(k, fontsize=9)
    for ax_ in axs.ravel(): ax_.axis('off')
    plt.suptitle(title); plt.tight_layout()
    return info

r = test_df.iloc[3]
info = show_stages(r.path, [r.x1, r.y1, r.x2, r.y2], CLASS_NAMES[r.label]); savefig('classical_stages'); plt.show()
print(info)
f = extract_classical_features(crop_box(cv2.imread(r.path), [r.x1, r.y1, r.x2, r.y2], CFG['CROP_PAD']))
print('Classical descriptor length:', f.shape[0])

In [ ]:
# @title 3.2 Cache padded GT crops (fast training) + extract classical features
from multiprocessing import Pool
def pool_map(fn, items, desc):
    with Pool(os.cpu_count()) as p:
        return list(tqdm(p.imap(fn, items, chunksize=16), total=len(items), desc=desc))

def _cache_one(args):
    src, dst, box = args
    if os.path.exists(dst): return
    crop = resize_max(crop_box(cv2.imread(src), box, CFG['CROP_PAD']), CFG['CACHE_MAX_SIDE'])
    cv2.imwrite(dst, crop, [cv2.IMWRITE_JPEG_QUALITY, 95])

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d = f'{WORK}/crops/{name}'; os.makedirs(d, exist_ok=True)
    df['crop_path'] = [f'{d}/{f}' for f in df.fname]
    pool_map(_cache_one, list(zip(df.path, df.crop_path, df[['x1', 'y1', 'x2', 'y2']].values.tolist())), f'cache {name}')

def classical_feats(df, name, boxes=None):
    fp = f'{WORK}/feat_{name}.npy'
    if os.path.exists(fp): return np.load(fp)
    if boxes is None:   # cached GT crops (already padded)
        items = [(p, None, 0.0) for p in df.crop_path]
    else:               # arbitrary boxes on originals (e.g. detector output)
        items = [(p, b, CFG['CROP_PAD']) for p, b in zip(df.path, boxes)]
    X = np.stack(pool_map(features_from_path, items, f'features {name}'))
    np.save(fp, X); return X

Xtr, Xval, Xte_gt = classical_feats(train_df, 'train'), classical_feats(val_df, 'val'), classical_feats(test_df, 'test_gt')
ytr, yval, yte = train_df.label.values, val_df.label.values, test_df.label.values
print(Xtr.shape, Xval.shape, Xte_gt.shape)

## 4. Classical-feature classifiers (baseline)
Standardise → PCA(256) → multinomial Logistic Regression (also used later for hybrid fusion) and RBF-SVM.

In [ ]:
# @title 4.1 Train + evaluate classical classifiers
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

CLASSICAL_PATH = f'{ART}/classical_lr.joblib'
if os.path.exists(CLASSICAL_PATH) and os.path.exists(f'{WORK}/svm.joblib'):
    bundle = joblib.load(CLASSICAL_PATH); svm = joblib.load(f'{WORK}/svm.joblib')
else:
    scaler = StandardScaler().fit(Xtr)
    pca = PCA(256, random_state=SEED).fit(scaler.transform(Xtr))
    Ztr = pca.transform(scaler.transform(Xtr))
    t0 = time.time(); lr = LogisticRegression(C=0.5, max_iter=2000).fit(Ztr, ytr); print(f'LR fit {time.time()-t0:.0f}s')
    t0 = time.time(); svm = SVC(C=10, kernel='rbf', gamma='scale', decision_function_shape='ovr').fit(Ztr, ytr); print(f'SVM fit {time.time()-t0:.0f}s')
    bundle = dict(scaler=scaler, pca=pca, lr=lr)
    joblib.dump(bundle, CLASSICAL_PATH); joblib.dump(svm, f'{WORK}/svm.joblib')
assert (bundle['lr'].classes_ == np.arange(NUM_CLASSES)).all()

def classical_logp(X):
    return bundle['lr'].predict_log_proba(bundle['pca'].transform(bundle['scaler'].transform(X)))
def classical_svm_scores(X):
    return svm.decision_function(bundle['pca'].transform(bundle['scaler'].transform(X)))

lp_val = classical_logp(Xval)
m_lr = cls_metrics(classical_logp(Xte_gt), yte); m_svm = cls_metrics(classical_svm_scores(Xte_gt), yte)

# latency: feature extraction + prediction per image
_samp = test_df.crop_path.values[:50]
t0 = time.time(); _f = np.stack([extract_classical_features(cv2.imread(p)) for p in _samp]); classical_logp(_f)
CLASSICAL_MS = (time.time() - t0) / len(_samp) * 1000
RESULTS['classical'] = dict(logreg_gt_crop=m_lr, svm_gt_crop=m_svm, ms_per_image=CLASSICAL_MS,
                            size_mb=os.path.getsize(CLASSICAL_PATH) / 1e6)
save_results()
print(pd.DataFrame({'Classical LogReg': m_lr, 'Classical RBF-SVM': m_svm}).T.round(4), f'\n{CLASSICAL_MS:.1f} ms/img (CPU)')

## 5. Vehicle detection / localisation
Three localisers compared on the **unseen test set** with IoU, localisation accuracy (IoU ≥ 0.5), mAP@0.5 and mAP@0.5:0.95:
1. **Classical** contour-based ROI (Section 3)
2. **YOLOv8s pretrained on COCO** (classes car / bus / truck), zero-shot
3. **YOLOv8s fine-tuned** on Stanford Cars boxes (single class `car`)

In [ ]:
# @title 5.1 Detection utilities (IoU, AP) + classical & pretrained YOLO
from ultralytics import YOLO

def iou_one(boxes, g):
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    ix1 = np.maximum(boxes[:, 0], g[0]); iy1 = np.maximum(boxes[:, 1], g[1])
    ix2 = np.minimum(boxes[:, 2], g[2]); iy2 = np.minimum(boxes[:, 3], g[3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    a = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]); b = (g[2] - g[0]) * (g[3] - g[1])
    return inter / (a + b - inter + 1e-9)

def average_precision(dets, gts, thr):
    recs = []
    for (b, s), g in zip(dets, gts):
        if len(b) == 0: continue
        ious = iou_one(b, g); matched = False
        for j in np.argsort(-s):
            tp = (not matched) and ious[j] >= thr
            matched |= tp; recs.append((s[j], tp))
    if not recs: return 0.0
    recs.sort(key=lambda r: -r[0]); tps = np.array([r[1] for r in recs], float)
    ctp, cfp = np.cumsum(tps), np.cumsum(1 - tps)
    rec, prec = ctp / len(gts), ctp / np.maximum(ctp + cfp, 1e-9)
    return float(np.mean([prec[rec >= t].max() if (rec >= t).any() else 0 for t in np.linspace(0, 1, 101)]))

def eval_localiser(dets, name, ms):
    top = [b[np.argmax(s)] if len(b) else None for b, s in dets]
    ious = np.array([iou_one(t, g)[0] if t is not None else 0.0 for t, g in zip(top, GT_TEST)])
    r = dict(mAP50=average_precision(dets, GT_TEST, 0.5),
             mAP50_95=float(np.mean([average_precision(dets, GT_TEST, t) for t in np.arange(0.5, 0.96, 0.05)])),
             mean_IoU=float(ious.mean()), loc_acc_IoU50=float((ious >= 0.5).mean()),
             detection_rate=float(np.mean([len(b) > 0 for b, _ in dets])), ms_per_image=ms)
    RESULTS.setdefault('detection', {})[name] = r; save_results()
    return r, ious

def yolo_detect(model, paths, classes=None, conf=0.01, bs=32):
    out = []; t0 = time.time()
    for i in tqdm(range(0, len(paths), bs), desc='detect', leave=False):
        res = model.predict(list(paths[i:i + bs]), conf=conf, iou=0.6, classes=classes,
                            imgsz=CFG['YOLO_IMGSZ'], half=DEVICE == 'cuda', verbose=False)
        out += [(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()) for r in res]
    return out, (time.time() - t0) / len(paths) * 1000

# 1) classical
t0 = time.time(); roi_res = pool_map(roi_from_path, list(test_df.path), 'classical ROI')
cls_ms = (time.time() - t0) / len(test_df) * 1000 * os.cpu_count()   # approx single-core ms/img
dets_classical = [(np.array([b]), np.array([s])) if b is not None else (np.zeros((0, 4)), np.zeros(0)) for b, s in roi_res]
TEST_BOXES_CLASSICAL = [list(b) if b is not None else None for b, _ in roi_res]
r_cls, iou_cls = eval_localiser(dets_classical, 'classical_contour_roi', cls_ms)

# 2) pretrained YOLO (COCO car=2, bus=5, truck=7)
det_pre = YOLO(CFG['YOLO_BASE'])
dets_pre, ms_pre = yolo_detect(det_pre, test_df.path.values, classes=[2, 5, 7])
r_pre, iou_pre = eval_localiser(dets_pre, 'yolov8s_coco_pretrained', ms_pre)
pd.DataFrame(RESULTS['detection']).T.round(4)

In [ ]:
# @title 5.2 Fine-tune YOLOv8 on Stanford Cars boxes
YD = f'{WORK}/yolo_ds'
for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    os.makedirs(f'{YD}/images/{split}', exist_ok=True); os.makedirs(f'{YD}/labels/{split}', exist_ok=True)
    for r in df.itertuples():
        dst = f'{YD}/images/{split}/{r.fname}'
        if not os.path.exists(dst): os.symlink(r.path, dst)
        cx, cy = (r.x1 + r.x2) / 2 / r.W, (r.y1 + r.y2) / 2 / r.H
        bw, bh = (r.x2 - r.x1) / r.W, (r.y2 - r.y1) / r.H
        with open(f'{YD}/labels/{split}/{r.fname[:-4]}.txt', 'w') as fh:
            fh.write(f'0 {min(max(cx,0),1):.6f} {min(max(cy,0),1):.6f} {min(bw,1):.6f} {min(bh,1):.6f}\n')
with open(f'{YD}/cars.yaml', 'w') as fh:
    fh.write(f'path: {YD}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: car\n')

YOLO_BEST = f'{ART}/yolo_car_best.pt'
if not os.path.exists(YOLO_BEST):
    det_train = YOLO(CFG['YOLO_BASE'])
    det_train.train(data=f'{YD}/cars.yaml', epochs=CFG['YOLO_EPOCHS'], imgsz=CFG['YOLO_IMGSZ'], batch=CFG['YOLO_BATCH'],
                    project=f'{OUT}/yolo_runs', name='car_det', exist_ok=True, patience=5, seed=SEED,
                    cos_lr=True, workers=NW, plots=True, verbose=False)
    shutil.copy(str(det_train.trainer.best), YOLO_BEST)
det_ft = YOLO(YOLO_BEST)

# official Ultralytics metrics on the unseen test split
m = det_ft.val(data=f'{YD}/cars.yaml', split='test', imgsz=CFG['YOLO_IMGSZ'], batch=32, half=DEVICE == 'cuda', plots=False, verbose=False)
RESULTS['detection_ultralytics_test'] = dict(mAP50=float(m.box.map50), mAP50_95=float(m.box.map),
                                             precision=float(m.box.mp), recall=float(m.box.mr))
dets_ft, ms_ft = yolo_detect(det_ft, test_df.path.values)
r_ft, iou_ft = eval_localiser(dets_ft, 'yolov8s_finetuned', ms_ft)

def choose_box(det, fallback_box):
    b, s = det
    if len(b): j = int(np.argmax(s)); return b[j].tolist()
    return fallback_box          # classical ROI, or None -> full image
TEST_BOXES_YOLO = [choose_box(d, fb) for d, fb in zip(dets_ft, TEST_BOXES_CLASSICAL)]
IOU_FT = np.array([iou_one(b, g)[0] if b is not None else 0 for b, g in zip(TEST_BOXES_YOLO, GT_TEST)])
DET_MS = ms_ft
print(RESULTS['detection_ultralytics_test'])
pd.DataFrame(RESULTS['detection']).T.round(4)

In [ ]:
# @title 5.3 Visual comparison of localisers (green=GT, blue=classical, yellow=COCO YOLO, red=fine-tuned YOLO)
fig, axs = plt.subplots(2, 4, figsize=(20, 8))
for ax_, i in zip(axs.ravel(), np.random.default_rng(3).choice(len(test_df), 8, replace=False)):
    im = cv2.cvtColor(cv2.imread(test_df.path[i]), cv2.COLOR_BGR2RGB)
    def rect(b, c, t=3):
        if b is not None: cv2.rectangle(im, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), c, t)
    rect(GT_TEST[i], (0, 255, 0), 5); rect(TEST_BOXES_CLASSICAL[i], (0, 0, 255))
    b, s = dets_pre[i]; rect(b[np.argmax(s)] if len(b) else None, (255, 255, 0)); rect(TEST_BOXES_YOLO[i], (255, 0, 0))
    ax_.imshow(im); ax_.set_title(f'IoU cls={iou_cls[i]:.2f} pre={iou_pre[i]:.2f} ft={iou_ft[i]:.2f}', fontsize=9); ax_.axis('off')
savefig('detection_examples'); plt.show()

## 6. Fine-grained classifier — ConvNeXt-Tiny transfer learning (196 classes)
ImageNet-22k pretrained ConvNeXt-Tiny, fine-tuned on padded vehicle ROIs. Augmentations target the advanced challenges:
- **Illumination**: colour jitter (brightness/contrast/saturation), grayscale
- **Viewing angle**: random perspective, rotation, horizontal flip, scale crops
- **Occlusion**: random erasing
- **Resolution**: random down-/up-sampling, blur

AdamW (head LR ×5), warm-up + cosine schedule, label smoothing, mixed precision. Resumable from Drive checkpoints.

In [ ]:
# @title 6.1 Data pipeline
cfg_m = timm.data.resolve_data_config({}, model=timm.create_model(CFG['CLS_MODEL'], pretrained=False))
MEAN, STD, S = list(cfg_m['mean']), list(cfg_m['std']), CFG['IMG_SIZE']

class RandomLowRes:
    def __init__(self, p=0.2, scales=(0.2, 0.6)): self.p, self.scales = p, scales
    def __call__(self, img):
        if random.random() < self.p:
            w, h = img.size; s = random.uniform(*self.scales)
            img = img.resize((max(8, int(w * s)), max(8, int(h * s))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return img

train_tf = T.Compose([
    T.Resize((S, S)),
    T.RandomResizedCrop(S, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.3, 0.03)], p=0.8),
    T.RandomGrayscale(0.05),
    T.RandomPerspective(distortion_scale=0.25, p=0.3),
    T.RandomApply([T.RandomRotation(10)], p=0.3),
    RandomLowRes(p=0.2),
    T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.15),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.35, scale=(0.02, 0.25), value='random'),
])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])

class CarDS(Dataset):
    """paths: image files. boxes: per-item xyxy box (crop with padding) or None (use whole image).
    enhance: apply classical adaptive illumination normalisation before the CNN."""
    def __init__(self, paths, labels, tf, boxes=None, pad=CFG['CROP_PAD'], enhance=False):
        self.paths, self.labels, self.tf, self.boxes, self.pad, self.enhance = list(paths), np.asarray(labels), tf, boxes, pad, enhance
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        bgr = cv2.imread(self.paths[i])
        if self.boxes is not None: bgr = crop_box(bgr, self.boxes[i], self.pad)
        if self.enhance: bgr, _ = adaptive_enhance(bgr, CFG['DARK_THR'], CFG['BRIGHT_THR'])
        return self.tf(Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))), int(self.labels[i])

train_ds = CarDS(train_df.crop_path, ytr, train_tf)
val_ds = CarDS(val_df.crop_path, yval, eval_tf)
train_loader = DataLoader(train_ds, CFG['BATCH'], shuffle=True, num_workers=NW, pin_memory=True, drop_last=True, persistent_workers=True)

x, _ = next(iter(DataLoader(train_ds, 8, shuffle=True)))
grid = (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1)
plt.figure(figsize=(18, 3)); plt.imshow(np.concatenate(list(grid.permute(0, 2, 3, 1).numpy()), 1)); plt.axis('off')
plt.title('Training augmentations'); savefig('augmentations'); plt.show()

In [ ]:
# @title 6.2 Train (resumable)
@torch.no_grad()
def get_logits(model, ds, bs=64, tta=False, desc='eval'):
    model.eval(); outs, outs_t = [], []
    for x, _ in tqdm(DataLoader(ds, bs, shuffle=False, num_workers=NW, pin_memory=True), desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            z = model(x).float()
            if tta: outs_t.append(((z + model(torch.flip(x, dims=[3])).float()) / 2).cpu())
        outs.append(z.cpu())
    z = torch.cat(outs).numpy()
    return (z, torch.cat(outs_t).numpy()) if tta else z

model = timm.create_model(CFG['CLS_MODEL'], pretrained=True, num_classes=NUM_CLASSES, drop_path_rate=0.1)
model = model.to(DEVICE).to(memory_format=torch.channels_last)
head_ids = {id(p) for p in model.get_classifier().parameters()}
opt = torch.optim.AdamW([
    {'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': CFG['LR']},
    {'params': list(model.get_classifier().parameters()), 'lr': CFG['LR'] * CFG['HEAD_LR_MULT']}],
    weight_decay=CFG['WEIGHT_DECAY'])
spe = len(train_loader); total = CFG['EPOCHS'] * spe; warm = CFG['WARMUP_EPOCHS'] * spe
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
try: scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == 'cuda')
crit = nn.CrossEntropyLoss(label_smoothing=CFG['LABEL_SMOOTH'])

LAST, BEST = f'{CKPT}/cls_last.pth', f'{ART}/classifier_best.pth'
start, best_acc, history = 0, 0.0, []
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    start, best_acc, history = ck['epoch'] + 1, ck['best_acc'], ck['history']
    print(f'Resumed from epoch {start}, best val acc {best_acc:.4f}')

for epoch in range(start, CFG['EPOCHS']):
    model.train(); t0 = time.time(); tl, n = 0.0, 0
    for x, y in tqdm(train_loader, desc=f'epoch {epoch+1}/{CFG["EPOCHS"]}', leave=False):
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(y); n += len(y)
    zv = get_logits(model, val_ds)
    mv = cls_metrics(zv, yval)
    history.append(dict(epoch=epoch + 1, train_loss=tl / n, val_top1=mv['top1'], val_top5=mv['top5'], minutes=(time.time() - t0) / 60))
    print(history[-1])
    if mv['top1'] > best_acc:
        best_acc = mv['top1']
        torch.save(dict(state_dict=model.state_dict(), model_name=CFG['CLS_MODEL'], num_classes=NUM_CLASSES,
                        img_size=S, mean=MEAN, std=STD, val_top1=best_acc, epoch=epoch + 1), BEST)
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                    epoch=epoch, best_acc=best_acc, history=history), LAST)

model.load_state_dict(torch.load(BEST, map_location=DEVICE, weights_only=False)['state_dict']); model.eval()
RESULTS['classifier_training'] = dict(history=history, best_val_top1=best_acc); save_results()
h = pd.DataFrame(history)
if len(h):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(h.epoch, h.train_loss); ax[0].set_title('Train loss')
    ax[1].plot(h.epoch, h.val_top1, label='top-1'); ax[1].plot(h.epoch, h.val_top5, label='top-5'); ax[1].legend(); ax[1].set_title('Validation accuracy')
    savefig('training_curves'); plt.show()

## 7. Evaluation on unseen test images (8,041) — deep model with different ROI sources
Same trained classifier fed with: whole image (no detection), classical-ROI crops, fine-tuned YOLO crops (± flip TTA, ± classical illumination normalisation), and ground-truth crops (upper bound).

In [ ]:
# @title 7.1 Test-set passes
Z = {}
Z['gt_crop'] = get_logits(model, CarDS(test_df.crop_path, yte, eval_tf), desc='GT crops')
Z['full_image'] = get_logits(model, CarDS(test_df.path, yte, eval_tf), desc='full image')
Z['classical_roi'] = get_logits(model, CarDS(test_df.path, yte, eval_tf, boxes=TEST_BOXES_CLASSICAL), desc='classical ROI')
Z['yolo_crop'], Z['yolo_crop_tta'] = get_logits(model, CarDS(test_df.path, yte, eval_tf, boxes=TEST_BOXES_YOLO), tta=True, desc='YOLO crops')
_, Z['yolo_crop_tta_enh'] = get_logits(model, CarDS(test_df.path, yte, eval_tf, boxes=TEST_BOXES_YOLO, enhance=True), tta=True, desc='YOLO+enhance')
zv, zv_tta = get_logits(model, val_ds, tta=True, desc='val')
np.savez(f'{WORK}/test_logits.npz', **Z)
pd.DataFrame({k: cls_metrics(v, yte) for k, v in Z.items()}).T.round(4)

## 8. Hybrid integration + ablation
**Hybrid pipeline** = fine-tuned YOLO ROI (classical-ROI fallback) → classical adaptive illumination normalisation → ConvNeXt (+flip TTA) → **late fusion with the classical colour/texture/HOG/geometry classifier**:
`score = log_softmax(z/T) + α · log p_classical`, then a second temperature `T2` for calibration. `T` and `α` are tuned on the held-out validation split only.

In [ ]:
# @title 8.1 Temperature, fusion weight, hybrid scores
def fit_temperature(logits, y):
    lt, yt = torch.tensor(logits, dtype=torch.float32), torch.tensor(y)
    logT = torch.zeros(1, requires_grad=True)
    o = torch.optim.LBFGS([logT], lr=0.1, max_iter=200)
    def closure():
        o.zero_grad(); l = F.cross_entropy(lt / logT.exp(), yt); l.backward(); return l
    o.step(closure); return float(logT.exp())

T_DEEP = fit_temperature(zv_tta, yval)
def fused(z, lp, alpha): return log_softmax(z / T_DEEP, 1) + alpha * lp
alphas = np.round(np.linspace(0, 1, 21), 2)
val_acc = [cls_metrics(fused(zv_tta, lp_val, a), yval)['top1'] for a in alphas]
ALPHA = float(alphas[int(np.argmax(val_acc))])
T2 = fit_temperature(fused(zv_tta, lp_val, ALPHA), yval)
print(f'T={T_DEEP:.3f}  alpha={ALPHA}  T2={T2:.3f}')
plt.figure(figsize=(6, 3)); plt.plot(alphas, val_acc, 'o-'); plt.xlabel('alpha (classical weight)'); plt.ylabel('val top-1'); plt.title('Fusion weight search'); savefig('fusion_alpha'); plt.show()

# classical features on YOLO crops of the test set (what the deployed system sees)
Xte_yolo = classical_feats(test_df, 'test_yolo', boxes=TEST_BOXES_YOLO)
lp_te_yolo = classical_logp(Xte_yolo)
def hybrid_probs(z, lp): return softmax(fused(z, lp, ALPHA) / T2, 1)
P_VAL = hybrid_probs(zv_tta, lp_val)
P_TEST = hybrid_probs(Z['yolo_crop_tta_enh'], lp_te_yolo)
RESULTS['calibration_params'] = dict(T=T_DEEP, alpha=ALPHA, T2=T2); save_results()

In [ ]:
# @title 8.2 Ablation table (each row adds / removes one component)
ablation = [
    ('Classical features only (LogReg) — YOLO ROI', lp_te_yolo),
    ('Classical features only (RBF-SVM) — YOLO ROI', classical_svm_scores(Xte_yolo)),
    ('Deep — whole image (no detection)', Z['full_image']),
    ('Deep — classical contour ROI', Z['classical_roi']),
    ('Deep — YOLO ROI', Z['yolo_crop']),
    ('Deep — YOLO ROI + flip TTA', Z['yolo_crop_tta']),
    ('Deep — YOLO ROI + TTA + illumination norm.', Z['yolo_crop_tta_enh']),
    ('HYBRID — above + classical feature fusion', P_TEST),
    ('Hybrid w/o illumination norm.', hybrid_probs(Z['yolo_crop_tta'], lp_te_yolo)),
    ('Hybrid w/o TTA', hybrid_probs(Z['yolo_crop'], lp_te_yolo)),
    ('Upper bound: deep on ground-truth crops', Z['gt_crop']),
]
abl = pd.DataFrame([dict(config=n, **cls_metrics(s, yte)) for n, s in ablation]).set_index('config')
RESULTS['ablation'] = abl.to_dict('index'); save_results()
abl.round(4)

In [ ]:
# @title 8.3 Classical vs Deep vs Hybrid — accuracy, size, latency
def bench(fn, n=30, warm=5):
    for _ in range(warm): fn()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n): fn()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000
x1 = torch.randn(1, 3, S, S, device=DEVICE).to(memory_format=torch.channels_last)
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
    CLS_MS = bench(lambda: model(x1)); CLS_TTA_MS = bench(lambda: (model(x1), model(torch.flip(x1, [3]))))
cls_size = os.path.getsize(BEST) / 1e6; yolo_size = os.path.getsize(YOLO_BEST) / 1e6; csize = RESULTS['classical']['size_mb']
cmp_rows = [
    dict(approach='Classical (HOG+colour+texture+geometry, LogReg)', **cls_metrics(lp_te_yolo, yte), size_mb=csize + yolo_size, ms_per_image=DET_MS + CLASSICAL_MS),
    dict(approach='Deep learning (YOLO ROI + ConvNeXt)', **cls_metrics(Z['yolo_crop'], yte), size_mb=cls_size + yolo_size, ms_per_image=DET_MS + CLS_MS),
    dict(approach='Hybrid (YOLO + classical preproc/fusion + ConvNeXt TTA)', **cls_metrics(P_TEST, yte), size_mb=cls_size + yolo_size + csize, ms_per_image=DET_MS + CLS_TTA_MS + CLASSICAL_MS),
]
cmp_df = pd.DataFrame(cmp_rows).set_index('approach')
RESULTS['approach_comparison'] = cmp_df.to_dict('index'); save_results()
ax = cmp_df[['top1', 'top5', 'make_acc']].plot.barh(figsize=(10, 3.5)); ax.set_xlim(0, 1); plt.title('Classical vs Deep vs Hybrid (unseen test)')
savefig('approach_comparison'); plt.show()
cmp_df.round(4)

## 9. Robustness — illumination, occlusion, resolution, viewing angle
A random subset of unseen test images is perturbed **before detection**, and the full system is re-run (detector + classifier), so the numbers reflect end-to-end robustness. Each condition reports detection IoU, deep top-1, hybrid top-1, and hybrid with/without classical illumination normalisation.

In [ ]:
# @title 9.1 Perturbation suite
def rotate_box(box, M):
    x1, y1, x2, y2 = box
    pts = np.array([[x1, y1, 1], [x2, y1, 1], [x1, y2, 1], [x2, y2, 1]]) @ M.T
    return [pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()]

def perturb(bgr, box, cond, rng):
    img = bgr.astype(np.float32); H, W = bgr.shape[:2]
    if cond == 'clean': return bgr, box
    if cond == 'dark': img = img * 0.3
    elif cond == 'very_dark': img = 255 * (img / 255) ** 3.0
    elif cond == 'overexposed': img = img * 1.6 + 60
    elif cond == 'low_contrast': img = (img - 128) * 0.35 + 128
    elif cond.startswith('occlusion'):
        frac = int(cond.split('_')[1]) / 100; x1, y1, x2, y2 = box; bw, bh = x2 - x1, y2 - y1
        a = rng.uniform(0.5, 2.0); ow = min(bw, math.sqrt(frac * bw * bh * a)); oh = min(bh, frac * bw * bh / ow)
        ox = rng.uniform(x1, x2 - ow); oy = rng.uniform(y1, y2 - oh)
        img[int(oy):int(oy + oh), int(ox):int(ox + ow)] = rng.integers(0, 255, 3)
    elif cond.startswith('lowres'):
        f = float(cond.split('_')[1])
        small = cv2.resize(bgr, (max(8, int(W * f)), max(8, int(H * f))), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR), box
    elif cond == 'motion_blur':
        k = np.zeros((15, 15), np.float32); k[7, :] = 1 / 15; img = cv2.filter2D(img, -1, k)
    elif cond == 'gauss_noise': img = img + rng.normal(0, 25, img.shape)
    elif cond == 'rotate_15':
        M = cv2.getRotationMatrix2D((W / 2, H / 2), 15, 1.0)
        return cv2.warpAffine(bgr, M, (W, H), borderMode=cv2.BORDER_REFLECT), rotate_box(box, M)
    return np.clip(img, 0, 255).astype(np.uint8), box

CONDITIONS = ['clean', 'dark', 'very_dark', 'overexposed', 'low_contrast', 'occlusion_25', 'occlusion_40',
              'lowres_0.25', 'lowres_0.1', 'motion_blur', 'gauss_noise', 'rotate_15']
rob_idx = np.random.default_rng(SEED).choice(len(test_df), min(CFG['N_ROBUST'], len(test_df)), replace=False)

@torch.no_grad()
def classify_crops(crops, enhance):
    xs = []
    for c in crops:
        if enhance: c, _ = adaptive_enhance(c, CFG['DARK_THR'], CFG['BRIGHT_THR'])
        xs.append(eval_tf(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))))
    x = torch.stack(xs).to(DEVICE).to(memory_format=torch.channels_last)
    with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
        return ((model(x).float() + model(torch.flip(x, [3])).float()) / 2).cpu().numpy()

rob_rows, rob_examples = [], {}
with Pool(os.cpu_count()) as pool:
    for cond in CONDITIONS:
        rng = np.random.default_rng(SEED); zs, zs_e, lps, ious, ys = [], [], [], [], []
        for k in tqdm(range(0, len(rob_idx), 32), desc=cond, leave=False):
            imgs, boxes, labs = [], [], []
            for i in rob_idx[k:k + 32]:
                bgr = cv2.imread(test_df.path[i]); s = min(1.0, 800 / max(bgr.shape[:2]))
                bgr = cv2.resize(bgr, None, fx=s, fy=s) if s < 1 else bgr
                im, bx = perturb(bgr, (GT_TEST[i] * s).tolist(), cond, rng)
                imgs.append(im); boxes.append(bx); labs.append(yte[i])
            res = det_ft.predict(imgs, conf=0.01, iou=0.6, imgsz=CFG['YOLO_IMGSZ'], half=DEVICE == 'cuda', verbose=False)
            crops = []
            for im, r, g in zip(imgs, res, boxes):
                b, sc = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
                pb = b[np.argmax(sc)].tolist() if len(b) else classical_roi(im)[0]
                ious.append(iou_one(pb, g)[0] if pb is not None else 0.0)
                crops.append(crop_box(im, pb, CFG['CROP_PAD']))
            zs.append(classify_crops(crops, False)); zs_e.append(classify_crops(crops, True))
            lps.append(classical_logp(np.stack(pool.map(extract_classical_features, crops))))
            ys += labs
            if cond not in rob_examples: rob_examples[cond] = cv2.cvtColor(imgs[0], cv2.COLOR_BGR2RGB)
        zc, zce, lpc, ys = np.concatenate(zs), np.concatenate(zs_e), np.concatenate(lps), np.array(ys)
        md, mh, mhe = cls_metrics(zc, ys), cls_metrics(hybrid_probs(zc, lpc), ys), cls_metrics(hybrid_probs(zce, lpc), ys)
        rob_rows.append(dict(condition=cond, mean_IoU=float(np.mean(ious)), loc_acc=float(np.mean(np.array(ious) >= 0.5)),
                             deep_top1=md['top1'], hybrid_top1=mh['top1'], hybrid_enh_top1=mhe['top1'], hybrid_enh_top5=mhe['top5']))
        print(rob_rows[-1])
rob_df = pd.DataFrame(rob_rows).set_index('condition')
RESULTS['robustness'] = rob_df.to_dict('index'); save_results()
rob_df.round(4)

In [ ]:
# @title 9.2 Robustness plots
ax = rob_df[['deep_top1', 'hybrid_top1', 'hybrid_enh_top1', 'loc_acc']].plot.bar(figsize=(16, 4.5)); ax.set_ylim(0, 1)
plt.title('End-to-end robustness (top-1 / localisation accuracy)'); plt.xticks(rotation=30); savefig('robustness_bars'); plt.show()
fig, axs = plt.subplots(2, 6, figsize=(22, 6))
for ax_, (c, im) in zip(axs.ravel(), rob_examples.items()):
    ax_.imshow(im); ax_.set_title(c); ax_.axis('off')
savefig('robustness_examples'); plt.show()

In [ ]:
# @title 9.3 Slice analysis on the clean test set: viewpoint proxy, resolution, brightness
pred = P_TEST.argmax(1); correct = pred == yte
ar = (GT_TEST[:, 2] - GT_TEST[:, 0]) / (GT_TEST[:, 3] - GT_TEST[:, 1])
area = np.sqrt((GT_TEST[:, 2] - GT_TEST[:, 0]) * (GT_TEST[:, 3] - GT_TEST[:, 1]))
TEST_BRIGHT = np.array([brightness(cv2.imread(p)) for p in tqdm(test_df.crop_path, desc='brightness', leave=False)])
slices = {
    'viewpoint (bbox aspect)': pd.cut(ar, [0, 1.3, 1.8, 10], labels=['front/rear (<1.3)', 'three-quarter (1.3-1.8)', 'side (>1.8)']),
    'vehicle resolution (sqrt px area)': pd.qcut(area, 4, labels=['smallest 25%', 'Q2', 'Q3', 'largest 25%']),
    'crop brightness (HSV-V)': pd.qcut(TEST_BRIGHT, 4, labels=['darkest 25%', 'Q2', 'Q3', 'brightest 25%']),
}
fig, axs = plt.subplots(1, 3, figsize=(18, 3.8)); RESULTS['slices'] = {}
for ax_, (name, bins) in zip(axs, slices.items()):
    s = pd.DataFrame({'bin': bins, 'acc': correct, 'iou': IOU_FT}).groupby('bin', observed=True).agg(top1=('acc', 'mean'), IoU=('iou', 'mean'), n=('acc', 'size'))
    RESULTS['slices'][name] = s.to_dict('index'); s[['top1', 'IoU']].plot.bar(ax=ax_, ylim=(0, 1), title=name, rot=15); print(name, '\n', s.round(3), '\n')
save_results(); savefig('slices'); plt.show()

## 10. Confidence analysis + low-confidence handling
- Reliability diagrams + Expected Calibration Error (before/after temperature scaling)
- Risk–coverage curve: accuracy of accepted predictions vs threshold
- Thresholds chosen **on validation** and applied to test:
  - `p_top1 ≥ τ` → **confident** (show make / model / year)
  - else if aggregated make probability `≥ τ_make` → **uncertain model** (show make + top-3 candidates)
  - else → **low confidence** (ask for a clearer image, show top-5 guesses)

In [ ]:
# @title 10.1 Calibration
def ece(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); bins = np.linspace(0, 1, n_bins + 1); e = 0; rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            acc, c = (pred[m] == y[m]).mean(), conf[m].mean(); e += m.mean() * abs(acc - c); rows.append((c, acc, m.sum()))
    return e, np.array(rows)
raw_p = softmax(Z['yolo_crop_tta_enh'], 1); ts_p = softmax(Z['yolo_crop_tta_enh'] / T_DEEP, 1)
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5)); RESULTS['ece'] = {}
for ax_, (n, p) in zip(axs, [('deep raw', raw_p), ('deep + temperature', ts_p), ('hybrid (final)', P_TEST)]):
    e, rows = ece(p, yte); RESULTS['ece'][n] = e
    ax_.plot([0, 1], [0, 1], 'k--'); ax_.bar(rows[:, 0], rows[:, 1], width=0.05, alpha=.7); ax_.set_title(f'{n}  ECE={e:.3f}'); ax_.set_xlabel('confidence'); ax_.set_ylabel('accuracy')
save_results(); savefig('reliability'); plt.show()

In [ ]:
# @title 10.2 Thresholds (from validation) + risk–coverage
def make_probs(P): return np.stack([P[:, MAKE_ID == m].sum(1) for m in range(len(MAKES))], 1)
def pick_threshold(conf, correct, target):
    for t in np.linspace(0, 0.99, 100):
        m = conf >= t
        if m.sum() >= 20 and correct[m].mean() >= target: return float(t)
    return 0.5
vc, vcor = P_VAL.max(1), P_VAL.argmax(1) == yval
MPV = make_probs(P_VAL)
TAU = pick_threshold(vc, vcor, CFG['TARGET_PRECISION'])
TAU_MAKE = pick_threshold(MPV.max(1), MPV.argmax(1) == MAKE_ID[yval], CFG['TARGET_PRECISION'])
print(f'tau={TAU:.2f}  tau_make={TAU_MAKE:.2f}')

tc, tcor = P_TEST.max(1), P_TEST.argmax(1) == yte
ts = np.linspace(0, 0.99, 100)
cov = [(tc >= t).mean() for t in ts]; accs = [tcor[tc >= t].mean() if (tc >= t).any() else np.nan for t in ts]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(ts, accs, label='accuracy of accepted'); ax[0].plot(ts, cov, label='coverage'); ax[0].axvline(TAU, c='r', ls='--', label=f'tau={TAU:.2f}')
ax[0].legend(); ax[0].set_xlabel('confidence threshold'); ax[0].set_title('Threshold analysis (test)')
ax[1].hist(tc[tcor], 40, alpha=.6, label='correct'); ax[1].hist(tc[~tcor], 40, alpha=.6, label='wrong'); ax[1].legend(); ax[1].set_title('Confidence distribution')
savefig('threshold_analysis'); plt.show()

MPT = make_probs(P_TEST)
status = np.where(tc >= TAU, 'confident', np.where(MPT.max(1) >= TAU_MAKE, 'uncertain_model', 'low_confidence'))
dec = pd.DataFrame({'status': status, 'model_correct': tcor, 'make_correct': MPT.argmax(1) == MAKE_ID[yte]}) \
        .groupby('status').agg(share=('model_correct', lambda s: len(s) / len(status)), model_acc=('model_correct', 'mean'), make_acc=('make_correct', 'mean'))
RESULTS['confidence'] = dict(tau=TAU, tau_make=TAU_MAKE, decision_breakdown=dec.to_dict('index'),
                             threshold_table={f'{t:.1f}': dict(coverage=float((tc >= t).mean()), acc=float(tcor[tc >= t].mean()) if (tc >= t).any() else None) for t in np.arange(0, 1, 0.1)})
save_results(); dec.round(4)

## 11. Model optimisation — FP16, ONNX, INT8 quantisation, pruning
Size, latency (batch 1) and top-1 on a fixed subset of unseen ground-truth crops.

In [ ]:
# @title 11.1 Optimisation benchmarks
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import torch.nn.utils.prune as prune

opt_idx = np.random.default_rng(1).choice(len(test_df), min(CFG['N_OPT_EVAL'], len(test_df)), replace=False)
opt_loader = DataLoader(CarDS(test_df.crop_path.values[opt_idx], yte[opt_idx], eval_tf), 50, num_workers=NW)
XO = [(x.half(), y.numpy()) for x, y in opt_loader]; yo = np.concatenate([y for _, y in XO])

@torch.no_grad()
def acc_torch(m, device, half=False):
    m = m.to(device).eval(); out = []
    for x, _ in XO:
        x = x.to(device); x = x.half() if half else x.float()
        out.append(m(x).float().cpu().numpy())
    return cls_metrics(np.concatenate(out), yo)['top1']

def acc_onnx(sess):
    return cls_metrics(np.concatenate([sess.run(None, {'input': x.float().numpy()})[0] for x, _ in XO]), yo)['top1']

rows = []
fp32 = copy.deepcopy(model).float().eval().to(memory_format=torch.contiguous_format)
torch.save(fp32.state_dict(), f'{WORK}/fp32.pth')
x1g = torch.randn(1, 3, S, S, device=DEVICE)
with torch.no_grad():
    rows.append(dict(variant='PyTorch FP32 (GPU)', size_mb=os.path.getsize(f'{WORK}/fp32.pth') / 1e6,
                     ms=bench(lambda: fp32.to(DEVICE)(x1g)), top1=acc_torch(fp32, DEVICE)))
    fp16 = copy.deepcopy(fp32).half().to(DEVICE)
    torch.save(dict(state_dict=fp16.state_dict(), model_name=CFG['CLS_MODEL'], num_classes=NUM_CLASSES, img_size=S, mean=MEAN, std=STD), f'{ART}/classifier_fp16.pth')
    rows.append(dict(variant='PyTorch FP16 (GPU, reduced precision)', size_mb=os.path.getsize(f'{ART}/classifier_fp16.pth') / 1e6,
                     ms=bench(lambda: fp16(x1g.half())), top1=acc_torch(fp16, DEVICE, half=True)))
    cpu_m = copy.deepcopy(fp32).cpu(); x1c = torch.randn(1, 3, S, S)
    rows.append(dict(variant='PyTorch FP32 (CPU)', size_mb=rows[0]['size_mb'], ms=bench(lambda: cpu_m(x1c), n=10, warm=2), top1=rows[0]['top1']))

ONNX32, ONNX8 = f'{ART}/classifier_fp32.onnx', f'{ART}/classifier_int8.onnx'
export_kw = dict(input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
try: torch.onnx.export(cpu_m, x1c, ONNX32, dynamo=False, **export_kw)
except TypeError: torch.onnx.export(cpu_m, x1c, ONNX32, **export_kw)
quantize_dynamic(ONNX32, ONNX8, weight_type=QuantType.QInt8, op_types_to_quantize=['MatMul', 'Gemm'])
for name, pth in [('ONNX Runtime FP32 (CPU)', ONNX32), ('ONNX Runtime INT8 dynamic quant (CPU)', ONNX8)]:
    sess = ort.InferenceSession(pth, providers=['CPUExecutionProvider']); xn = x1c.numpy()
    rows.append(dict(variant=name, size_mb=os.path.getsize(pth) / 1e6, ms=bench(lambda: sess.run(None, {'input': xn}), n=20, warm=3), top1=acc_onnx(sess)))

for amount in [0.3, 0.5]:
    pm = copy.deepcopy(fp32)
    params = [(m_, 'weight') for m_ in pm.modules() if isinstance(m_, (nn.Conv2d, nn.Linear))]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    for m_, _ in params: prune.remove(m_, 'weight')
    nz = sum((m_.weight != 0).sum().item() for m_, _ in params); tot = sum(m_.weight.numel() for m_, _ in params)
    rows.append(dict(variant=f'L1 global pruning {int(amount*100)}% (no fine-tune)', size_mb=rows[0]['size_mb'] * nz / tot,
                     ms=np.nan, top1=acc_torch(pm, DEVICE), sparsity=1 - nz / tot))

def yolo_ms(half):
    im = cv2.imread(test_df.path[0])
    return bench(lambda: det_ft.predict(im, imgsz=CFG['YOLO_IMGSZ'], half=half, verbose=False), n=20, warm=3)
rows.append(dict(variant='YOLOv8s detector FP32', size_mb=os.path.getsize(YOLO_BEST) / 1e6, ms=yolo_ms(False), top1=np.nan))
rows.append(dict(variant='YOLOv8s detector FP16', size_mb=os.path.getsize(YOLO_BEST) / 1e6 / 2, ms=yolo_ms(True), top1=np.nan))
opt_df = pd.DataFrame(rows).set_index('variant')
RESULTS['optimization'] = opt_df.to_dict('index'); save_results()
opt_df.round(4)

## 12. Failure cases
Wrong predictions of the final hybrid system are tagged automatically with their most likely cause (localisation error, low resolution, poor lighting, truncation/occlusion, year confusion, same-make confusion, cross-make look-alike). Grad-CAM shows where the network looked.

In [ ]:
# @title 12.1 Cause tagging + most-confused pairs
from collections import Counter
BODY = ['Sedan', 'SUV', 'Coupe', 'Convertible', 'Hatchback', 'Wagon', 'Minivan', 'Van', 'Cab', 'Supercab', 'Hatchback']
def body_type(c):
    for b in BODY:
        if b in CLASS_META[c]['model']: return b
    return 'other'
min_side = np.minimum(GT_TEST[:, 2] - GT_TEST[:, 0], GT_TEST[:, 3] - GT_TEST[:, 1])
truncated = (GT_TEST[:, 0] <= 2) | (GT_TEST[:, 1] <= 2) | (GT_TEST[:, 2] >= test_df.W.values - 3) | (GT_TEST[:, 3] >= test_df.H.values - 3)

def failure_cause(i, p):
    g = yte[i]; G, P = CLASS_META[g], CLASS_META[p]
    if IOU_FT[i] < 0.5: return 'localisation', f'Detector box IoU={IOU_FT[i]:.2f}: the crop missed part of the car or included other objects, so the classifier saw an incomplete vehicle.'
    if min_side[i] < 100: return 'low_resolution', f'Vehicle is only ~{int(min_side[i])}px on its short side; fine details (badges, grille, lamp shapes) that separate classes are lost.'
    if TEST_BRIGHT[i] < 70: return 'low_light', f'Crop brightness {TEST_BRIGHT[i]:.0f}/255: under-exposure hides texture and colour cues even after CLAHE.'
    if truncated[i]: return 'truncation/occlusion', 'Vehicle is cut off by the image border, so the distinguishing front/rear parts are partially missing.'
    if G['make'] == P['make'] and G['model'] == P['model']: return 'year_confusion', f'Same {G["make"]} {G["model"]}, different year ({G["year"]} vs {P["year"]}): generations differ only in subtle facelift details.'
    if G['make'] == P['make']: return 'same_make', f'Both are {G["make"]} vehicles sharing brand design language (grille, headlights); difference lies in proportions/body details.'
    return 'cross_make_lookalike', f'Different makes but similar silhouette ({body_type(g)} vs {body_type(p)}) and colour; viewpoint hides brand-specific cues.'

pred = P_TEST.argmax(1); conf = P_TEST.max(1); wrong = np.where(pred != yte)[0]
causes = [failure_cause(i, pred[i]) for i in wrong]
cnt = Counter(c for c, _ in causes)
RESULTS['failure_causes'] = dict(cnt); save_results()
plt.figure(figsize=(8, 3)); plt.barh(list(cnt.keys()), list(cnt.values())); plt.title(f'Failure causes ({len(wrong)} errors / {len(yte)})'); savefig('failure_causes'); plt.show()
pairs = Counter((CLASS_NAMES[yte[i]], CLASS_NAMES[pred[i]]) for i in wrong).most_common(12)
pd.DataFrame([dict(true=a, predicted=b, count=c) for (a, b), c in pairs])

In [ ]:
# @title 12.2 Failure gallery with Grad-CAM
class GradCAM:
    def __init__(self, m):
        self.m = m; layer = m.stages[-1] if hasattr(m, 'stages') else list(m.children())[-3]
        layer.register_forward_hook(self._fwd)
    def _fwd(self, mod, inp, out):
        self.act = out
        if out.requires_grad: out.register_hook(lambda g: setattr(self, 'grad', g))
    def __call__(self, x, cls=None):
        self.m.zero_grad(); x = x.clone().requires_grad_(True)
        out = self.m(x); cls = int(out.argmax(1)) if cls is None else cls
        out[0, cls].backward()
        w = self.grad.mean((2, 3), keepdim=True); cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, x.shape[-2:], mode='bilinear')[0, 0].detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)

cam_model = copy.deepcopy(model).float().eval().to(memory_format=torch.contiguous_format); gcam = GradCAM(cam_model)
def cam_overlay(crop_bgr):
    x = eval_tf(Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)))[None].to(DEVICE)
    cam = gcam(x); rgb = cv2.cvtColor(cv2.resize(crop_bgr, (S, S)), cv2.COLOR_BGR2RGB)
    heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return (0.55 * rgb + 0.45 * heat).astype(np.uint8)

by_cause = {}
for j in np.argsort(-conf[wrong]):           # most confident mistakes first, diverse causes
    i = wrong[j]; c, why = failure_cause(i, pred[i])
    if len(by_cause.setdefault(c, [])) < 2: by_cause[c].append((i, why))
picked = [v for vs in by_cause.values() for v in vs][:6]
fig, axs = plt.subplots(len(picked), 3, figsize=(16, 4.2 * len(picked))); axs = np.atleast_2d(axs)
RESULTS['failure_examples'] = []
for row, (i, why) in zip(axs, picked):
    im = cv2.imread(test_df.path[i]); vis = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).copy()
    g, b = GT_TEST[i], TEST_BOXES_YOLO[i]
    cv2.rectangle(vis, (int(g[0]), int(g[1])), (int(g[2]), int(g[3])), (0, 255, 0), 4)
    if b is not None: cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 3)
    ex = train_df[train_df.label == pred[i]].crop_path.values[0]
    row[0].imshow(vis); row[0].set_title(f'GT: {CLASS_NAMES[yte[i]]}', fontsize=9)
    row[1].imshow(cam_overlay(crop_box(im, b, CFG['CROP_PAD']))); row[1].set_title(f'Pred: {CLASS_NAMES[pred[i]]} ({conf[i]:.0%}) — Grad-CAM', fontsize=9)
    row[2].imshow(cv2.cvtColor(cv2.imread(ex), cv2.COLOR_BGR2RGB)); row[2].set_title('Training example of predicted class', fontsize=9)
    row[0].set_xlabel(why, fontsize=8, wrap=True)
    for a in row: a.set_xticks([]); a.set_yticks([])
    RESULTS['failure_examples'].append(dict(file=test_df.fname[i], true=CLASS_NAMES[yte[i]], pred=CLASS_NAMES[pred[i]], conf=float(conf[i]), cause=failure_cause(i, pred[i])[0], explanation=why))
plt.tight_layout(); savefig('failure_gallery'); plt.show(); save_results()
for e in RESULTS['failure_examples']: print(f"- [{e['cause']}] {e['true']} -> {e['pred']} ({e['conf']:.0%}): {e['explanation']}")

## 13. End-to-end system for the frontend
Writes `config.json`, `class_meta.json` and **`inference.py`** into `artifacts/`. Your frontend (Streamlit / Gradio / FastAPI) only needs:
```python
from inference import VehicleRecognizer
rec = VehicleRecognizer('artifacts')            # or backend='onnx' for CPU-only INT8
out = rec.predict('car.jpg', gradcam=True)
out['vehicles']        # [{make, model, year, confidence, status, message, top5, box, det_conf, ...}]
out['annotated_image'] # RGB numpy with boxes + labels
out['stages']          # {stage name: RGB image} — intermediate classical-CV results
out['classical_info']  # brightness, aspect ratio, edge density, symmetry, ...
out['timings_ms']
```
`metrics.json` + `plots/` hold everything for the "compare classical / deep / hybrid" page.

In [ ]:
# @title 13.1 Save config + class metadata
CONFIG = dict(model_name=CFG['CLS_MODEL'], num_classes=NUM_CLASSES, img_size=S, mean=MEAN, std=STD, crop_pad=CFG['CROP_PAD'],
              T=T_DEEP, alpha=ALPHA, T2=T2, tau=TAU, tau_make=TAU_MAKE, det_conf=CFG['DET_CONF'], yolo_imgsz=CFG['YOLO_IMGSZ'],
              enhance=True, dark_thr=CFG['DARK_THR'], bright_thr=CFG['BRIGHT_THR'], tta=True)
json.dump(CONFIG, open(f'{ART}/config.json', 'w'), indent=2)
json.dump(CLASS_META, open(f'{ART}/class_meta.json', 'w'), indent=1)
open(f'{ART}/requirements.txt', 'w').write('torch\ntorchvision\ntimm\nultralytics\nopencv-python-headless\nscikit-image\nscikit-learn\njoblib\nnumpy\nPillow\nonnxruntime\n')
CONFIG

In [ ]:
%%writefile /content/work/inference.py
"""P5 Fine-Grained Vehicle Recognition — inference module for the frontend.

    from inference import VehicleRecognizer
    rec = VehicleRecognizer('artifacts')          # backend='onnx' -> CPU INT8 classifier
    out = rec.predict('car.jpg')

Needs in the artifacts folder: config.json, class_meta.json, yolo_car_best.pt, classifier_best.pth
(or classifier_int8.onnx), classical_lr.joblib, and classical_cv.py next to this file.
"""
import os, json, time
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
from classical_cv import to_bgr, adaptive_enhance, crop_box, classical_roi, extract_classical_features, processing_stages


def _log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))


STATUS_COLORS = {'confident': (0, 200, 0), 'uncertain_model': (255, 165, 0), 'low_confidence': (230, 0, 0)}


class VehicleRecognizer:
    def __init__(self, art_dir='artifacts', device=None, backend='torch', onnx_file='classifier_int8.onnx', use_fusion=True):
        from ultralytics import YOLO
        self.art = art_dir
        self.cfg = json.load(open(os.path.join(art_dir, 'config.json')))
        self.classes = json.load(open(os.path.join(art_dir, 'class_meta.json')))
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.backend = backend
        self.det = YOLO(os.path.join(art_dir, 'yolo_car_best.pt'))
        S = self.cfg['img_size']
        self.tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(self.cfg['mean'], self.cfg['std'])])
        self.clf = None
        if backend == 'onnx':
            import onnxruntime as ort
            self.sess = ort.InferenceSession(os.path.join(art_dir, onnx_file), providers=['CPUExecutionProvider'])
        else:
            import timm
            ck = torch.load(os.path.join(art_dir, 'classifier_best.pth'), map_location='cpu', weights_only=False)
            self.clf = timm.create_model(ck['model_name'], pretrained=False, num_classes=ck['num_classes'])
            self.clf.load_state_dict(ck['state_dict'])
            self.clf.eval().to(self.device)
        self.classical = None
        p = os.path.join(art_dir, 'classical_lr.joblib')
        if use_fusion and self.cfg.get('alpha', 0) > 0 and os.path.exists(p):
            import joblib
            self.classical = joblib.load(p)
        self.make_names = sorted({c['make'] for c in self.classes})
        self.make_idx = np.array([self.make_names.index(c['make']) for c in self.classes])
        self._cam = None

    # ---------- detection with low-confidence fallbacks ----------
    def detect(self, bgr, max_vehicles=5):
        r = self.det.predict(bgr, conf=0.05, iou=0.6, imgsz=self.cfg['yolo_imgsz'], verbose=False)[0]
        b, s = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
        order = np.argsort(-s)
        keep = [i for i in order if s[i] >= self.cfg['det_conf']][:max_vehicles]
        if keep:
            return [(b[i].tolist(), float(s[i]), 'yolo') for i in keep]
        if len(b):
            i = order[0]
            return [(b[i].tolist(), float(s[i]), 'yolo_low_confidence')]
        box, score, _ = classical_roi(bgr)
        if box is not None:
            return [(list(map(float, box)), score, 'classical_roi_fallback')]
        H, W = bgr.shape[:2]
        return [([0.0, 0.0, float(W), float(H)], 0.0, 'full_image_fallback')]

    # ---------- classification ----------
    def _logits(self, crops):
        x = torch.stack([self.tf(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops])
        if self.backend == 'onnx':
            xn = x.numpy()
            z = self.sess.run(None, {'input': xn})[0]
            if self.cfg.get('tta', True):
                z = (z + self.sess.run(None, {'input': np.ascontiguousarray(xn[:, :, :, ::-1])})[0]) / 2
            return z
        x = x.to(self.device)
        with torch.no_grad():
            z = self.clf(x).float()
            if self.cfg.get('tta', True):
                z = (z + self.clf(torch.flip(x, [3])).float()) / 2
        return z.cpu().numpy()

    def classify(self, crops):
        enh = [adaptive_enhance(c, self.cfg['dark_thr'], self.cfg['bright_thr'])[0] if self.cfg.get('enhance') else c for c in crops]
        s = _log_softmax(self._logits(enh) / self.cfg['T'])
        if self.classical is not None:
            f = np.stack([extract_classical_features(c) for c in crops])
            b = self.classical
            s = s + self.cfg['alpha'] * b['lr'].predict_log_proba(b['pca'].transform(b['scaler'].transform(f)))
        return np.exp(_log_softmax(s / self.cfg['T2']))

    def decide(self, p):
        top = np.argsort(-p)[:5]
        c = self.classes[top[0]]
        mp = np.bincount(self.make_idx, weights=p, minlength=len(self.make_names))
        mk = int(mp.argmax())
        conf = float(p[top[0]])
        cands = ', '.join(self.classes[i]['name'] for i in top[:3])
        if conf >= self.cfg['tau']:
            status, msg = 'confident', f"{c['make']} {c['model']} ({c['year']}) — {conf:.1%}"
        elif mp[mk] >= self.cfg['tau_make']:
            status, msg = 'uncertain_model', f"Likely a {self.make_names[mk]} ({mp[mk]:.0%}); exact model/year uncertain. Candidates: {cands}"
        else:
            status, msg = 'low_confidence', f"Low confidence — try a clearer / closer image. Top guesses: {cands}"
        return dict(label=c['name'], make=c['make'], model=c['model'], year=c['year'], class_id=int(top[0]),
                    confidence=conf, status=status, message=msg,
                    make_guess=self.make_names[mk], make_confidence=float(mp[mk]),
                    top5=[dict(label=self.classes[i]['name'], confidence=float(p[i])) for i in top])

    # ---------- explainability ----------
    def gradcam(self, crop_bgr):
        if self.clf is None:
            return None
        if self._cam is None:
            layer = self.clf.stages[-1] if hasattr(self.clf, 'stages') else list(self.clf.children())[-3]
            def hook(mod, inp, out):
                self._act = out
                if out.requires_grad:
                    out.register_hook(lambda g: setattr(self, '_grad', g))
            layer.register_forward_hook(hook)
            self._cam = True
        x = self.tf(Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)))[None].to(self.device)
        self.clf.zero_grad()
        with torch.enable_grad():
            out = self.clf(x.requires_grad_(True))
            out[0, int(out.argmax(1))].backward()
        w = self._grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * self._act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=crop_bgr.shape[:2], mode='bilinear')[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
        return cv2.cvtColor((0.55 * crop_bgr + 0.45 * heat).astype(np.uint8), cv2.COLOR_BGR2RGB)

    # ---------- drawing ----------
    @staticmethod
    def draw(bgr, vehicles):
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).copy()
        th = max(2, int(max(rgb.shape[:2]) / 300))
        for v in vehicles:
            x1, y1, x2, y2 = [int(t) for t in v['box']]
            col = STATUS_COLORS[v['status']]
            cv2.rectangle(rgb, (x1, y1), (x2, y2), col, th)
            txt = f"{v['label']} {v['confidence']:.0%}" if v['status'] == 'confident' else f"{v['make_guess']}? {v['make_confidence']:.0%}"
            fs = max(0.5, th / 3)
            (tw, tht), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, fs, max(1, th // 2))
            yt = max(tht + 6, y1)
            cv2.rectangle(rgb, (x1, yt - tht - 6), (x1 + tw + 6, yt), col, -1)
            cv2.putText(rgb, txt, (x1 + 3, yt - 4), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), max(1, th // 2), cv2.LINE_AA)
        return rgb

    # ---------- full pipeline ----------
    def predict(self, image, max_vehicles=5, return_stages=True, gradcam=False):
        t0 = time.time()
        bgr = to_bgr(image)
        dets = self.detect(bgr, max_vehicles)
        t1 = time.time()
        crops = [crop_box(bgr, d[0], self.cfg['crop_pad']) for d in dets]
        probs = self.classify(crops)
        t2 = time.time()
        vehicles = []
        for (box, dconf, src), p in zip(dets, probs):
            v = self.decide(p)
            v.update(box=[float(t) for t in box], det_conf=dconf, det_source=src)
            vehicles.append(v)
        out = dict(vehicles=vehicles, annotated_image=self.draw(bgr, vehicles),
                   timings_ms=dict(detection=(t1 - t0) * 1000, classification=(t2 - t1) * 1000))
        if return_stages:
            out['stages'], out['classical_info'] = processing_stages(bgr, dets[0][0])
        if gradcam:
            out['gradcam'] = self.gradcam(crops[0])
        out['timings_ms']['total'] = (time.time() - t0) * 1000
        return out

In [ ]:
# @title 13.2 Test the exported pipeline (exactly what the frontend will run)
shutil.copy(f'{WORK}/inference.py', ART)
import inference; importlib.reload(inference)
from inference import VehicleRecognizer
rec = VehicleRecognizer(ART)

def show_result(out, title=''):
    fig, axs = plt.subplots(1, 3 if 'gradcam' in out and out['gradcam'] is not None else 2, figsize=(18, 5))
    axs[0].imshow(out['annotated_image']); axs[0].set_title(title); axs[0].axis('off')
    v = out['vehicles'][0]
    axs[1].barh([t['label'] for t in v['top5']][::-1], [t['confidence'] for t in v['top5']][::-1]); axs[1].set_xlim(0, 1)
    axs[1].set_title(f"{v['status']}: {v['message']}", fontsize=9)
    if len(axs) == 3: axs[2].imshow(out['gradcam']); axs[2].set_title('Grad-CAM'); axs[2].axis('off')
    plt.tight_layout(); plt.show()

lat = []
for i in [0, 10, 20]:
    out = rec.predict(test_df.path[i], gradcam=True); lat.append(out['timings_ms']['total'])
    show_result(out, f'GT: {CLASS_NAMES[yte[i]]}')
    print({k: v for k, v in out['vehicles'][0].items() if k != 'top5'}, out['timings_ms'])
RESULTS['end_to_end_ms_per_image'] = float(np.mean(lat[1:])); save_results()
rec_onnx = VehicleRecognizer(ART, backend='onnx')
print('ONNX INT8 backend:', rec_onnx.predict(test_df.path[0], return_stages=False)['vehicles'][0]['message'])

In [ ]:
# @title 13.3 Upload your own image (demo of the full application output)
try:
    from google.colab import files
    up = files.upload()
    for fn in up:
        out = rec.predict(fn, gradcam=True)
        show_result(out, fn)
        st = out['stages']; fig, axs = plt.subplots(3, 4, figsize=(18, 10))
        for a, (k, v) in zip(axs.ravel(), st.items()): a.imshow(v); a.set_title(k, fontsize=9)
        for a in axs.ravel(): a.axis('off')
        plt.suptitle('Intermediate image-processing results'); plt.show()
        print(out['classical_info'])
        for v in out['vehicles']: print(v['message'])
except Exception as e:
    print('Upload skipped:', e)

## 14. Export artifacts
Everything is already in Drive (`MyDrive/P5_vehicle_recognition/artifacts`). This also zips it for direct download.

| File | Use |
|---|---|
| `yolo_car_best.pt` | fine-tuned YOLOv8s vehicle detector |
| `classifier_best.pth` | ConvNeXt-Tiny FP32 weights (+ model name, mean/std, img size) |
| `classifier_fp16.pth`, `classifier_fp32.onnx`, `classifier_int8.onnx` | optimised variants |
| `classical_lr.joblib` | scaler + PCA + LogReg on classical features (hybrid fusion) |
| `config.json`, `class_meta.json` | temperatures, α, thresholds, make/model/year table |
| `classical_cv.py`, `inference.py`, `requirements.txt` | drop-in backend for the frontend |
| `metrics.json` (+ `../plots/`) | all evaluation numbers and charts |

In [ ]:
# @title 14.1 Zip + download
print(json.dumps({k: RESULTS[k] for k in ['approach_comparison', 'detection_ultralytics_test', 'confidence'] if k in RESULTS}, indent=1, default=_js)[:3000])
shutil.make_archive(f'{OUT}/P5_artifacts', 'zip', ART)
shutil.make_archive(f'{OUT}/P5_plots', 'zip', PLOTS)
for f in os.listdir(ART): print(f'{f:28s} {os.path.getsize(f"{ART}/{f}")/1e6:8.2f} MB')
try:
    from google.colab import files
    files.download(f'{OUT}/P5_artifacts.zip'); files.download(f'{OUT}/P5_plots.zip')
except Exception as e:
    print('Download manually from Drive:', OUT, e)

In [ ]:
# @title 14.2 (Optional) quick Gradio preview of the app — prototype before building your own frontend
!pip -q install gradio
import gradio as gr
def gr_predict(img):
    out = rec.predict(img[:, :, ::-1].copy(), gradcam=True)
    v = out['vehicles'][0]
    stages = [(im, name) for name, im in out['stages'].items()]
    return out['annotated_image'], {t['label']: t['confidence'] for t in v['top5']}, v['message'], out['gradcam'], stages, out['classical_info']
gr.Interface(gr_predict, gr.Image(type='numpy'),
             [gr.Image(label='Detection'), gr.Label(label='Top-5'), gr.Textbox(label='Result'), gr.Image(label='Grad-CAM'),
              gr.Gallery(label='Intermediate classical CV'), gr.JSON(label='Classical features')],
             title='P5 Fine-Grained Vehicle Recognition').launch(share=True, debug=False)